# Federated Learning with **Progressive** Hard Sample Removal

## Strategy Overview
This notebook implements **progressive hard sample removal** - identifying and removing problematic samples incrementally throughout training:

### Progressive Removal Strategy:
1. **Every 10 Rounds**: Analyze sample performance over the last 10 rounds
2. **Incremental Removal**: Remove worst 2.5% of samples each cycle (5 cycles total)
3. **Quick Detection**: Hard samples reveal themselves within 10 rounds through:
   - Consistently high loss (top 15% worst)
   - Low prediction confidence (< 45%)
   - Frequent prediction changes (high variance)
4. **Adaptive Approach**: Continuously clean data as training progresses

### Why This Works Better Than Profiling:
- ⚡ **Fast Detection**: Truly bad samples show up immediately (within 10 rounds)
- 🎯 **Conservative**: Only 2.5% removed per cycle = 12.5% total over 50 rounds
- 🔄 **Adaptive**: Later cycles catch samples that become problematic as model improves
- 💾 **Data Preservation**: Maintains per-class minimums, never over-removes

### Hard Sample Criteria (Combined Score):
- **High Loss** (40% weight): Consistently high loss in recent rounds
- **Low Confidence** (40% weight): Model rarely confident about these samples
- **Prediction Variance** (20% weight): Predictions keep changing

### Configuration:
- 100 clients, each with 100 samples (10 per class)
- 100 total rounds with 5 removal cycles (rounds 10, 20, 30, 40, 50)
- Remove 2.5% per cycle → ~12.5% total removal (~12-13 samples per client)
- Min 6 samples per class preserved

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from collections import defaultdict
import os
from tqdm import tqdm

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.10.0


In [2]:
# GPU Configuration
print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU detected and configured")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("⚠ No GPU detected - Running on CPU")
print("=" * 60 + "\n")

GPU CONFIGURATION
TensorFlow version: 2.10.0
Num GPUs Available: 1
✓ GPU detected and configured



## Configuration

In [3]:
# Federated Learning Configuration
NUM_CLIENTS = 100
NUM_ROUNDS = 100  # Total rounds with progressive removal
LOCAL_EPOCHS = 10
BATCH_SIZE = 32
NUM_CLASSES = 10

# Progressive Removal Parameters
REMOVAL_INTERVAL = 10  # Analyze and remove samples every N rounds
REMOVAL_PER_CYCLE = 0.025  # Remove worst 2.5% per cycle
NUM_REMOVAL_CYCLES = 5  # Total of 5 removal cycles (rounds 10, 20, 30, 40, 50)
ANALYSIS_WINDOW = 10  # Analyze last 10 rounds of data
MIN_SAMPLES_PER_CLASS = 6  # Never go below 6 samples per class

# Detection Thresholds (more aggressive for quick detection)
HIGH_LOSS_PERCENTILE = 0.85  # Top 15% worst losses
LOW_CONFIDENCE_THRESHOLD = 0.45  # Confidence below 45%
HIGH_VARIANCE_THRESHOLD = 0.25  # Prediction variance threshold

# Directories
DATA_DIR = 'mnist_100_clients'
RESULTS_DIR = 'results_progressive_hard_sample_removal'
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 60)
print("FEDERATED LEARNING - PROGRESSIVE HARD SAMPLE REMOVAL")
print("=" * 60)
print(f"Number of Clients: {NUM_CLIENTS}")
print(f"Total Rounds: {NUM_ROUNDS}")
print(f"Local Epochs per Round: {LOCAL_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"\nProgressive Removal Strategy:")
print(f"  Removal Interval: Every {REMOVAL_INTERVAL} rounds")
print(f"  Removal per Cycle: {REMOVAL_PER_CYCLE*100:.1f}%")
print(f"  Number of Cycles: {NUM_REMOVAL_CYCLES}")
print(f"  Total Removal: ~{REMOVAL_PER_CYCLE*NUM_REMOVAL_CYCLES*100:.1f}%")
print(f"  Removal Rounds: {[i*REMOVAL_INTERVAL for i in range(1, NUM_REMOVAL_CYCLES+1)]}")
print(f"  Analysis Window: Last {ANALYSIS_WINDOW} rounds")
print(f"  Min Samples per Class: {MIN_SAMPLES_PER_CLASS}")
print(f"\nDetection Criteria:")
print(f"  High Loss: Top {(1-HIGH_LOSS_PERCENTILE)*100:.0f}% worst")
print(f"  Low Confidence: < {LOW_CONFIDENCE_THRESHOLD*100:.0f}%")
print(f"  High Variance: > {HIGH_VARIANCE_THRESHOLD}")
print(f"\nData Directory: {DATA_DIR}/")
print(f"Results Directory: {RESULTS_DIR}/")
print("=" * 60 + "\n")

FEDERATED LEARNING - PROGRESSIVE HARD SAMPLE REMOVAL
Number of Clients: 100
Total Rounds: 100
Local Epochs per Round: 10
Batch Size: 32

Progressive Removal Strategy:
  Removal Interval: Every 10 rounds
  Removal per Cycle: 2.5%
  Number of Cycles: 5
  Total Removal: ~12.5%
  Removal Rounds: [10, 20, 30, 40, 50]
  Analysis Window: Last 10 rounds
  Min Samples per Class: 6

Detection Criteria:
  High Loss: Top 15% worst
  Low Confidence: < 45%
  High Variance: > 0.25

Data Directory: mnist_100_clients/
Results Directory: results_progressive_hard_sample_removal/



## Load Data

In [4]:
# Load test data (common for all clients)
print("Loading common test dataset...")
test_file = os.path.join(DATA_DIR, 'test_500_samples.npz')
test_data = np.load(test_file)

x_test = test_data['x'] / 255.0
y_test = test_data['y']
x_test = x_test.reshape(len(x_test), 28*28)

print(f"✓ Test data loaded: {x_test.shape}")
print(f"  Labels shape: {y_test.shape}")

Loading common test dataset...
✓ Test data loaded: (500, 784)
  Labels shape: (500,)


In [5]:
# Load all client data
print(f"\nLoading data for {NUM_CLIENTS} clients...")
client_data = []

for client_id in range(1, NUM_CLIENTS + 1):
    client_file = os.path.join(DATA_DIR, f'client_{client_id}.npz')
    data = np.load(client_file)
    
    x_client = data['x'] / 255.0
    y_client = data['y']
    x_client = x_client.reshape(len(x_client), 28*28)
    
    client_data.append({
        'x_train': x_client,
        'y_train': y_client,
        'x_test': x_test,
        'y_test': y_test,
        'x_train_original': x_client.copy(),  # Keep original for comparison
        'y_train_original': y_client.copy()
    })
    
    if client_id % 20 == 0:
        print(f"  Loaded {client_id}/{NUM_CLIENTS} clients")

print(f"\n✓ All {NUM_CLIENTS} clients loaded successfully")
print(f"  Each client has {len(client_data[0]['x_train'])} training samples")
print(f"  Common test set: {len(x_test)} samples")


Loading data for 100 clients...
  Loaded 20/100 clients
  Loaded 40/100 clients
  Loaded 60/100 clients
  Loaded 80/100 clients
  Loaded 100/100 clients

✓ All 100 clients loaded successfully
  Each client has 100 training samples
  Common test set: 500 samples


## Model Architecture

In [6]:
# Define model
def create_model():
    """Lightweight model optimized for small datasets"""
    model = keras.Sequential([
        keras.layers.Dense(64, input_shape=(784,), activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation="softmax")
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Test model creation
print("Testing model architecture...")
test_model = create_model()
test_model.summary()
print("\n✓ Model architecture validated")

Testing model architecture...
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                50240     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_2 (Dense)             (None, 10)                330       
                                                                 
Total params: 52,650
Trainable params: 52,650
Non-trainable params: 0
_________________________________________________________________

✓ Model architecture 

## Hard Sample Detection Functions

In [7]:
# Functions for detecting hard samples
def compute_sample_loss(model, x_samples, y_samples):
    """
    Compute loss for each individual sample
    Returns: array of losses, one per sample
    """
    predictions = model.predict(x_samples, verbose=0)
    losses = []
    
    for i in range(len(x_samples)):
        # Compute categorical crossentropy for this sample
        true_class = int(y_samples[i])
        pred_prob = predictions[i][true_class]
        loss = -np.log(pred_prob + 1e-7)  # Add small epsilon to avoid log(0)
        losses.append(loss)
    
    return np.array(losses)

def compute_sample_confidence(model, x_samples):
    """
    Compute prediction confidence for each sample
    Returns: array of confidences (max probability), one per sample
    """
    predictions = model.predict(x_samples, verbose=0)
    confidences = np.max(predictions, axis=1)
    return confidences

def compute_prediction_variance(predictions_history):
    """
    Compute how much predictions vary across rounds for each sample
    predictions_history: list of prediction arrays (one per round)
    Returns: variance score for each sample
    """
    if len(predictions_history) < 2:
        return np.zeros(predictions_history[0].shape[0])
    
    # Stack predictions: shape (rounds, samples, classes)
    stacked = np.array(predictions_history)
    
    # Get predicted class for each round
    pred_classes = np.argmax(stacked, axis=2)  # shape: (rounds, samples)
    
    # Compute variance of predicted classes
    variances = []
    for sample_idx in range(pred_classes.shape[1]):
        sample_preds = pred_classes[:, sample_idx]
        # Use standard deviation of predictions as inconsistency measure
        variance = np.std(sample_preds) / NUM_CLASSES  # Normalize by number of classes
        variances.append(variance)
    
    return np.array(variances)

def identify_hard_samples(losses_history, confidences_history, predictions_history, 
                          y_train, removal_percentage, min_per_class):
    """
    Identify hard samples based on multiple criteria from recent rounds
    Returns: boolean mask (True = keep sample, False = remove sample)
    """
    num_samples = len(y_train)
    
    if len(losses_history) == 0:
        # No data yet, keep all samples
        return np.ones(num_samples, dtype=bool), np.zeros(num_samples)
    
    # Compute statistics across recent rounds
    avg_loss = np.mean(losses_history, axis=0)
    avg_confidence = np.mean(confidences_history, axis=0)
    pred_variance = compute_prediction_variance(predictions_history)
    
    # Normalize scores to [0, 1] with more aggressive thresholds
    loss_score = (avg_loss - avg_loss.min()) / (avg_loss.max() - avg_loss.min() + 1e-7)
    conf_score = 1.0 - avg_confidence  # Lower confidence = higher score
    var_score = pred_variance / (pred_variance.max() + 1e-7) if pred_variance.max() > 0 else np.zeros(num_samples)
    
    # Combined hardness score (weighted average)
    hardness_score = 0.4 * loss_score + 0.4 * conf_score + 0.2 * var_score
    
    # Determine how many samples to remove
    num_to_remove = int(num_samples * removal_percentage)
    
    # Get indices sorted by hardness (highest = hardest)
    sorted_indices = np.argsort(hardness_score)[::-1]
    
    # Initialize mask (True = keep)
    keep_mask = np.ones(num_samples, dtype=bool)
    
    # Try to remove hardest samples, but respect per-class minimums
    samples_per_class = defaultdict(list)
    for idx, label in enumerate(y_train):
        samples_per_class[int(label)].append(idx)
    
    removed_count = 0
    for idx in sorted_indices:
        if removed_count >= num_to_remove:
            break
        
        # Check if removing this sample would violate per-class minimum
        label = int(y_train[idx])
        current_count = sum([keep_mask[i] for i in samples_per_class[label]])
        
        if current_count > min_per_class:
            keep_mask[idx] = False
            removed_count += 1
    
    return keep_mask, hardness_score

print("✓ Hard sample detection functions defined")

✓ Hard sample detection functions defined


## Federated Learning Functions

In [8]:
# Federated Averaging function
def federated_averaging(weights_list):
    """Average weights from all clients (FedAvg)"""
    avg_weights = []
    for weights_tuple in zip(*weights_list):
        avg_weights.append(np.mean(weights_tuple, axis=0))
    return avg_weights

print("✓ Federated averaging function defined")

✓ Federated averaging function defined


## Initialize Federated Learning

In [9]:
# Initialize global model
print("\n" + "=" * 60)
print("INITIALIZING FEDERATED LEARNING")
print("=" * 60)

global_model = create_model()
global_weights = global_model.get_weights()

# Tracking arrays
client_train_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_test_acc_history = [[] for _ in range(NUM_CLIENTS)]
client_best_weights = [None for _ in range(NUM_CLIENTS)]
client_best_test_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_best_train_acc = [0.0 for _ in range(NUM_CLIENTS)]
client_rejections = [[] for _ in range(NUM_CLIENTS)]

# Hard sample tracking (rolling window for progressive removal)
client_sample_losses = [[] for _ in range(NUM_CLIENTS)]  # Rolling window of loss arrays
client_sample_confidences = [[] for _ in range(NUM_CLIENTS)]  # Rolling window of confidence arrays
client_sample_predictions = [[] for _ in range(NUM_CLIENTS)]  # Rolling window of prediction arrays
client_removed_samples = [0 for _ in range(NUM_CLIENTS)]  # Cumulative count of removed samples
client_removal_history = [[] for _ in range(NUM_CLIENTS)]  # Track removals per cycle
removal_rounds = []  # Track which rounds had removals

print("✓ Global model initialized")
print("✓ Tracking arrays created")
print("✓ Progressive hard sample removal enabled")
print(f"✓ Removal scheduled for rounds: {[i*REMOVAL_INTERVAL for i in range(1, NUM_REMOVAL_CYCLES+1)]}")


INITIALIZING FEDERATED LEARNING
✓ Global model initialized
✓ Tracking arrays created
✓ Progressive hard sample removal enabled
✓ Removal scheduled for rounds: [10, 20, 30, 40, 50]


## Progressive Training with Incremental Sample Removal

In [ ]:
# Main training loop with progressive sample removal
print("\n" + "=" * 60)
print("PROGRESSIVE FEDERATED TRAINING WITH HARD SAMPLE REMOVAL")
print("=" * 60 + "\n")

for round_num in tqdm(range(NUM_ROUNDS), desc="Training Rounds", unit="round"):
    print(f"\n{'='*60}")
    print(f"ROUND {round_num + 1}/{NUM_ROUNDS}")
    print(f"{'='*60}")
    
    local_weights = []
    round_train_accs = [0.0] * NUM_CLIENTS
    round_test_accs = [0.0] * NUM_CLIENTS
    round_rejections = 0
    round_acceptances = 0
    
    # Check if this is a removal round
    is_removal_round = (round_num + 1) % REMOVAL_INTERVAL == 0 and (round_num + 1) <= NUM_REMOVAL_CYCLES * REMOVAL_INTERVAL
    
    # Train all clients
    for client_id in tqdm(range(NUM_CLIENTS), desc=f"Training Clients (Round {round_num + 1})", 
                         unit="client", leave=False):
        
        client_model = create_model()
        client_model.set_weights(global_weights)
        
        x_train = client_data[client_id]['x_train']
        y_train = client_data[client_id]['y_train']
        x_test = client_data[client_id]['x_test']
        y_test = client_data[client_id]['y_test']
        
        # Train with current data
        history = client_model.fit(
            x_train, y_train,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        
        # Collect sample-level statistics for analysis
        sample_losses = compute_sample_loss(client_model, x_train, y_train)
        sample_confidences = compute_sample_confidence(client_model, x_train)
        sample_predictions = client_model.predict(x_train, verbose=0)
        
        # Maintain rolling window (keep last ANALYSIS_WINDOW rounds)
        client_sample_losses[client_id].append(sample_losses)
        client_sample_confidences[client_id].append(sample_confidences)
        client_sample_predictions[client_id].append(sample_predictions)
        
        if len(client_sample_losses[client_id]) > ANALYSIS_WINDOW:
            client_sample_losses[client_id] = client_sample_losses[client_id][-ANALYSIS_WINDOW:]
            client_sample_confidences[client_id] = client_sample_confidences[client_id][-ANALYSIS_WINDOW:]
            client_sample_predictions[client_id] = client_sample_predictions[client_id][-ANALYSIS_WINDOW:]
        
        # Evaluate
        train_acc = history.history['accuracy'][-1]
        _, test_acc = client_model.evaluate(x_test, y_test, verbose=0)
        
        # Weight acceptance/rejection logic
        trained_weights = client_model.get_weights()
        
        if test_acc > client_best_test_acc[client_id]:
            client_best_weights[client_id] = [w.copy() for w in trained_weights]
            client_best_test_acc[client_id] = test_acc
            client_best_train_acc[client_id] = train_acc
            client_rejections[client_id].append(0)
            round_acceptances += 1
        else:
            if client_best_weights[client_id] is None:
                client_best_weights[client_id] = [w.copy() for w in trained_weights]
                client_best_test_acc[client_id] = test_acc
                client_best_train_acc[client_id] = train_acc
                client_rejections[client_id].append(0)
                round_acceptances += 1
            else:
                client_rejections[client_id].append(1)
                round_rejections += 1
                test_acc = client_best_test_acc[client_id]
                train_acc = client_best_train_acc[client_id]
        
        # Store for averaging
        local_weights.append(client_best_weights[client_id])
        
        # Store accuracies
        client_train_acc_history[client_id].append(train_acc)
        client_test_acc_history[client_id].append(test_acc)
        round_train_accs[client_id] = train_acc
        round_test_accs[client_id] = test_acc
    
    # Federated averaging
    global_weights = federated_averaging(local_weights)
    global_model.set_weights(global_weights)
    
    # Progressive sample removal at specified intervals
    if is_removal_round:
        print(f"\n🗑️  REMOVAL CYCLE {(round_num + 1) // REMOVAL_INTERVAL}/{NUM_REMOVAL_CYCLES}")
        print("=" * 60)
        
        total_removed_this_cycle = 0
        
        for client_id in range(NUM_CLIENTS):
            x_train = client_data[client_id]['x_train']
            y_train = client_data[client_id]['y_train']
            
            samples_before = len(y_train)
            
            # Identify hard samples based on recent performance
            keep_mask, hardness_scores = identify_hard_samples(
                client_sample_losses[client_id],
                client_sample_confidences[client_id],
                client_sample_predictions[client_id],
                y_train,
                REMOVAL_PER_CYCLE,
                MIN_SAMPLES_PER_CLASS
            )
            
            # Apply mask to remove hard samples
            x_train_cleaned = x_train[keep_mask]
            y_train_cleaned = y_train[keep_mask]
            
            # Update client data
            client_data[client_id]['x_train'] = x_train_cleaned
            client_data[client_id]['y_train'] = y_train_cleaned
            
            # Reset statistics for new dataset
            client_sample_losses[client_id] = []
            client_sample_confidences[client_id] = []
            client_sample_predictions[client_id] = []
            
            samples_after = len(y_train_cleaned)
            removed_this_cycle = samples_before - samples_after
            client_removed_samples[client_id] += removed_this_cycle
            client_removal_history[client_id].append(removed_this_cycle)
            total_removed_this_cycle += removed_this_cycle
        
        removal_rounds.append(round_num + 1)
        print(f"✓ Removed {total_removed_this_cycle} samples across all clients")
        print(f"   Avg per client: {total_removed_this_cycle/NUM_CLIENTS:.1f}")
        print(f"   Cumulative total: {sum(client_removed_samples)} samples")
        print("=" * 60)
    
    # Round summary
    avg_train_acc = np.mean(round_train_accs) * 100
    avg_test_acc = np.mean(round_test_accs) * 100
    min_test_acc = np.min(round_test_accs) * 100
    max_test_acc = np.max(round_test_accs) * 100
    
    print(f"\n📊 Round {round_num + 1} Summary:")
    print(f"   Avg Training Accuracy: {avg_train_acc:.2f}%")
    print(f"   Avg Test Accuracy: {avg_test_acc:.2f}%")
    print(f"   Test Accuracy Range: [{min_test_acc:.2f}%, {max_test_acc:.2f}%]")
    print(f"   Weights Accepted: {round_acceptances}, Rejected: {round_rejections}")
    if is_removal_round:
        print(f"   🗑️  Samples Removed This Round: {total_removed_this_cycle}")

print("\n" + "="*60)
print("PROGRESSIVE TRAINING COMPLETE!")
print("="*60 + "\n")


PROGRESSIVE FEDERATED TRAINING WITH HARD SAMPLE REMOVAL



Training Rounds:   0%|          | 0/100 [00:00<?, ?round/s]


ROUND 1/100


Training Rounds:   1%|          | 1/100 [01:01<1:41:52, 61.74s/round]


📊 Round 1 Summary:
   Avg Training Accuracy: 56.66%
   Avg Test Accuracy: 56.74%
   Test Accuracy Range: [45.20%, 65.40%]
   Weights Accepted: 100, Rejected: 0

ROUND 2/100


...


📊 Round 87 Summary:
   Avg Training Accuracy: 96.40%
   Avg Test Accuracy: 89.43%
   Test Accuracy Range: [86.80%, 91.20%]
   Weights Accepted: 3, Rejected: 97

ROUND 88/100


Training Rounds:  88%|████████▊ | 88/100 [1:40:24<15:25, 77.13s/round]


📊 Round 88 Summary:
   Avg Training Accuracy: 96.40%
   Avg Test Accuracy: 89.43%
   Test Accuracy Range: [86.80%, 91.20%]
   Weights Accepted: 0, Rejected: 100

ROUND 89/100


Training Rounds:  89%|████████▉ | 89/100 [1:41:39<14:02, 76.55s/round]


📊 Round 89 Summary:
   Avg Training Accuracy: 96.40%
   Avg Test Accuracy: 89.44%
   Test Accuracy Range: [86.80%, 91.20%]
   Weights Accepted: 3, Rejected: 97

ROUND 90/100


## Progressive Removal Summary

In [ ]:
# Summarize progressive removal statistics
print("\n" + "=" * 60)
print("PROGRESSIVE REMOVAL SUMMARY")
print("=" * 60 + "\n")

total_samples_original = NUM_CLIENTS * 100  # Assuming 100 samples per client initially
total_samples_final = sum([len(client_data[i]['y_train']) for i in range(NUM_CLIENTS)])
total_removed = sum(client_removed_samples)

print(f"Removal Cycles Executed: {len(removal_rounds)}")
print(f"Removal Rounds: {removal_rounds}")
print(f"\nOverall Statistics:")
print(f"  Original Total Samples: {total_samples_original}")
print(f"  Final Total Samples: {total_samples_final}")
print(f"  Total Samples Removed: {total_removed}")
print(f"  Overall Removal Rate: {(total_removed/total_samples_original)*100:.2f}%")
print(f"\nPer-Client Statistics:")
print(f"  Avg Samples Removed: {np.mean(client_removed_samples):.2f}")
print(f"  Min Removed: {min(client_removed_samples)}")
print(f"  Max Removed: {max(client_removed_samples)}")
print(f"  Std Dev: {np.std(client_removed_samples):.2f}")

# Show removal per cycle
if len(removal_rounds) > 0:
    print(f"\nRemoval Per Cycle:")
    for cycle_idx in range(min(len(removal_rounds), NUM_REMOVAL_CYCLES)):
        cycle_removals = [client_removal_history[i][cycle_idx] if cycle_idx < len(client_removal_history[i]) else 0 
                         for i in range(NUM_CLIENTS)]
        total_cycle = sum(cycle_removals)
        print(f"  Cycle {cycle_idx+1} (Round {removal_rounds[cycle_idx]}): {total_cycle} samples ({total_cycle/NUM_CLIENTS:.1f} avg per client)")

# Show per-class breakdown for sample client
sample_client_id = 0
y_original = client_data[sample_client_id]['y_train_original']
y_final = client_data[sample_client_id]['y_train']
print(f"\n📋 Sample Client 1 - Per-Class Breakdown:")
for class_idx in range(NUM_CLASSES):
    before = np.sum(y_original == class_idx)
    after = np.sum(y_final == class_idx)
    print(f"   Class {class_idx}: {before} → {after} ({before-after} removed)")

print("=" * 60 + "\n")

## Results Analysis

In [ ]:
# Calculate final accuracies and statistics
final_train_accs = [client_train_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]
final_test_accs = [client_test_acc_history[i][-1] * 100 for i in range(NUM_CLIENTS)]

avg_final_train = np.mean(final_train_accs)
avg_final_test = np.mean(final_test_accs)

# Calculate improvement from first removal
if len(removal_rounds) > 0:
    first_removal_round = removal_rounds[0] - 1  # Index before first removal
    before_removal_accs = [client_test_acc_history[i][first_removal_round] * 100 for i in range(NUM_CLIENTS)]
    avg_before_removal = np.mean(before_removal_accs)
else:
    avg_before_removal = final_test_accs[0]

improvement = avg_final_test - avg_before_removal

# Rejection statistics
total_rejections_per_client = [sum(client_rejections[i]) for i in range(NUM_CLIENTS)]

print("=" * 60)
print("FINAL RESULTS - PROGRESSIVE HARD SAMPLE REMOVAL")
print("=" * 60)
print(f"Overall Performance:")
print(f"  Average Final Training Accuracy: {avg_final_train:.2f}%")
print(f"  Average Final Test Accuracy: {avg_final_test:.2f}%")
print(f"  Test Accuracy Range: [{np.min(final_test_accs):.2f}%, {np.max(final_test_accs):.2f}%]")
print(f"  Test Accuracy Std Dev: {np.std(final_test_accs):.2f}%")

print(f"\nImprovement Analysis:")
if len(removal_rounds) > 0:
    print(f"  Avg Accuracy Before First Removal: {avg_before_removal:.2f}%")
    print(f"  Avg Accuracy After Progressive Cleaning: {avg_final_test:.2f}%")
    print(f"  Improvement: {improvement:+.2f}% ({'+' if improvement > 0 else ''}{(improvement/avg_before_removal)*100:.2f}% relative)")
else:
    print(f"  No removals performed")

print(f"\nSample Removal Statistics:")
print(f"  Total Samples Removed: {sum(client_removed_samples)}")
print(f"  Avg Samples Removed per Client: {np.mean(client_removed_samples):.2f}")
print(f"  Overall Removal Rate: {(sum(client_removed_samples)/total_samples_original)*100:.2f}%")

print(f"\nWeight Rejection Statistics:")
print(f"  Total Weight Rejections: {sum(total_rejections_per_client)}")
print(f"  Rejection Rate: {sum(total_rejections_per_client) / (NUM_CLIENTS * NUM_ROUNDS) * 100:.2f}%")
print("=" * 60)

## Visualizations

In [ ]:
# Plot 1: Test Accuracy for All Clients with Progressive Removal Marking
print("Creating test accuracy visualization...")

fig, axes = plt.subplots(10, 10, figsize=(30, 30))
fig.suptitle('Test Accuracy - Progressive Hard Sample Removal (100 Clients)', 
             fontsize=20, fontweight='bold', y=0.995)

rounds = range(1, NUM_ROUNDS + 1)

for client_id in range(NUM_CLIENTS):
    row = client_id // 10
    col = client_id % 10
    ax = axes[row, col]
    
    test_accs = [acc * 100 for acc in client_test_acc_history[client_id]]
    final_acc = test_accs[-1]
    improvement = final_acc - test_accs[0]
    removed = client_removed_samples[client_id]
    
    # Color based on number of samples removed
    if removed >= 15:
        color = '#FF6B6B'  # Red for high removal
        title_color = 'red'
    elif removed >= 10:
        color = '#FFA500'  # Orange for medium removal
        title_color = 'orange'
    else:
        color = '#2E86AB'  # Blue for low removal
        title_color = 'blue'
    
    ax.plot(rounds, test_accs, linewidth=2, color=color, alpha=0.8)
    ax.fill_between(rounds, test_accs, alpha=0.3, color=color)
    
    # Mark removal rounds
    for removal_round in removal_rounds:
        ax.axvline(x=removal_round, color='red', linestyle='--', alpha=0.4, linewidth=1)
    
    ax.set_title(f'C{client_id+1} ({improvement:+.1f}%) [-{removed}]', 
                fontsize=8, fontweight='bold', color=title_color)
    
    ax.text(NUM_ROUNDS, final_acc, f'{final_acc:.0f}%', 
            fontsize=7, fontweight='bold', verticalalignment='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, 100)
    ax.set_xlim(0.5, NUM_ROUNDS + 0.5)
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_xticklabels(['1', '25', '50', '75', '100'], fontsize=7)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_yticklabels(['0', '25', '50', '75', '100'], fontsize=7)
    
    if row == 9:
        ax.set_xlabel('Round', fontsize=8)
    if col == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=8)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'test_accuracy_progressive_removal.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_path}")
plt.show()

In [ ]:
# Plot 2: Average Accuracy with Progressive Removal Markers
print("Creating average accuracy comparison...")

rounds = range(1, NUM_ROUNDS + 1)
avg_test_per_round = [np.mean([client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]) 
                      for r in range(NUM_ROUNDS)]
min_test = [np.min([client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]) 
            for r in range(NUM_ROUNDS)]
max_test = [np.max([client_test_acc_history[i][r] * 100 for i in range(NUM_CLIENTS)]) 
            for r in range(NUM_ROUNDS)]

fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(rounds, avg_test_per_round, marker='o', linewidth=2.5, markersize=5, 
        color='#2E86AB', label='Average Test Accuracy', alpha=0.9, markevery=max(1, NUM_ROUNDS//10))
ax.plot(rounds, min_test, linestyle='--', linewidth=1.5, 
        color='gray', label='Min/Max Range', alpha=0.5)
ax.plot(rounds, max_test, linestyle='--', linewidth=1.5, 
        color='gray', alpha=0.5)

ax.fill_between(rounds, min_test, max_test, alpha=0.1, color='gray')

# Mark removal rounds
for idx, removal_round in enumerate(removal_rounds):
    ax.axvline(x=removal_round, color='red', linestyle='--', alpha=0.6, linewidth=2)
    if idx == 0:
        ax.text(removal_round, 5, f'↓ Cycle {idx+1}', fontsize=8, ha='center', 
               color='red', fontweight='bold')
    else:
        ax.text(removal_round, 5 + (idx % 2) * 3, f'↓ {idx+1}', fontsize=8, ha='center', 
               color='red', fontweight='bold')

ax.text(NUM_ROUNDS/2, 95, f'Progressive Removal\n{len(removal_rounds)} cycles, {sum(client_removed_samples)} total samples', 
       fontsize=10, ha='center', bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.6))

ax.set_xlabel('Communication Round', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Federated Learning - Progressive Hard Sample Removal\nTest Accuracy Over Time', 
            fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_ylim(0, 100)

plt.tight_layout()
avg_plot_path = os.path.join(RESULTS_DIR, 'average_accuracy_comparison.png')
plt.savefig(avg_plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {avg_plot_path}")
plt.show()

In [ ]:
# Plot 3: Sample Removal Distribution and Analysis
print("Creating sample removal distribution...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 3a: Histogram of samples removed per client
ax1 = axes[0]
removal_counts = [client_removed_samples[i] for i in range(NUM_CLIENTS)]
ax1.hist(removal_counts, bins=20, color='#FF6B6B', alpha=0.7, edgecolor='black')
ax1.axvline(np.mean(removal_counts), color='blue', linestyle='--', linewidth=2, 
           label=f'Mean: {np.mean(removal_counts):.1f}')
ax1.set_xlabel('Number of Samples Removed', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Clients', fontsize=11, fontweight='bold')
ax1.set_title('Distribution of Sample Removal', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Plot 3b: Scatter plot - samples removed vs final accuracy
ax2 = axes[1]
scatter = ax2.scatter(removal_counts, final_accs, c=final_accs, cmap='RdYlGn', 
                     s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Number of Samples Removed', fontsize=11, fontweight='bold')
ax2.set_ylabel('Final Test Accuracy (%)', fontsize=11, fontweight='bold')
ax2.set_title('Sample Removal vs Final Accuracy', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Accuracy (%)', fontsize=10)

# Calculate correlation
correlation = np.corrcoef(removal_counts, final_accs)[0, 1]
ax2.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
        transform=ax2.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
dist_plot_path = os.path.join(RESULTS_DIR, 'sample_removal_distribution.png')
plt.savefig(dist_plot_path, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {dist_plot_path}")
plt.show()

print("\n" + "="*60)
print("ALL VISUALIZATIONS COMPLETE!")
print("="*60)

## Save Results

In [ ]:
# Save results summary
results_file = os.path.join(RESULTS_DIR, 'final_results_progressive_removal.txt')

with open(results_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("FEDERATED LEARNING - PROGRESSIVE HARD SAMPLE REMOVAL\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("Configuration:\n")
    f.write(f"  Number of Clients: {NUM_CLIENTS}\n")
    f.write(f"  Total Rounds: {NUM_ROUNDS}\n")
    f.write(f"  Removal Interval: Every {REMOVAL_INTERVAL} rounds\n")
    f.write(f"  Removal per Cycle: {REMOVAL_PER_CYCLE*100:.1f}%\n")
    f.write(f"  Number of Cycles: {NUM_REMOVAL_CYCLES}\n")
    f.write(f"  Removal Rounds: {removal_rounds}\n")
    f.write(f"  Local Epochs: {LOCAL_EPOCHS}\n")
    f.write(f"  Min Samples per Class: {MIN_SAMPLES_PER_CLASS}\n\n")
    
    f.write("Overall Performance:\n")
    f.write(f"  Average Final Test Accuracy: {avg_final_test:.2f}%\n")
    f.write(f"  Test Accuracy Range: [{np.min(final_test_accs):.2f}%, {np.max(final_test_accs):.2f}%]\n")
    f.write(f"  Standard Deviation: {np.std(final_test_accs):.2f}%\n\n")
    
    f.write("Improvement Analysis:\n")
    if len(removal_rounds) > 0:
        f.write(f"  Avg Accuracy Before First Removal: {avg_before_removal:.2f}%\n")
        f.write(f"  Avg Accuracy After Progressive Cleaning: {avg_final_test:.2f}%\n")
        f.write(f"  Absolute Improvement: {improvement:+.2f}%\n")
        f.write(f"  Relative Improvement: {(improvement/avg_before_removal)*100:+.2f}%\n\n")
    else:
        f.write(f"  No removals performed\n\n")
    
    f.write("Sample Removal Statistics:\n")
    f.write(f"  Original Total Samples: {total_samples_original}\n")
    f.write(f"  Final Total Samples: {total_samples_final}\n")
    f.write(f"  Total Samples Removed: {total_removed}\n")
    f.write(f"  Overall Removal Rate: {(total_removed/total_samples_original)*100:.2f}%\n")
    f.write(f"  Avg Removed per Client: {np.mean(client_removed_samples):.2f}\n")
    f.write(f"  Min Removed: {min(client_removed_samples)}\n")
    f.write(f"  Max Removed: {max(client_removed_samples)}\n\n")
    
    f.write("Top 10 Clients by Final Accuracy:\n")
    top_clients = sorted(enumerate(final_test_accs), key=lambda x: x[1], reverse=True)[:10]
    for rank, (client_id, acc) in enumerate(top_clients, 1):
        removed = client_removed_samples[client_id]
        f.write(f"  {rank}. Client {client_id+1}: {acc:.2f}% (removed {removed} samples)\n")
    
    f.write("\nBottom 10 Clients by Final Accuracy:\n")
    bottom_clients = sorted(enumerate(final_test_accs), key=lambda x: x[1])[:10]
    for rank, (client_id, acc) in enumerate(bottom_clients, 1):
        removed = client_removed_samples[client_id]
        f.write(f"  {rank}. Client {client_id+1}: {acc:.2f}% (removed {removed} samples)\n")
    
    f.write("\n" + "=" * 60 + "\n")

print(f"✓ Results saved to: {results_file}")
print("\n" + "="*60)
print("ALL TASKS COMPLETE!")
print("="*60)